# Bronze: POS events (Auto Loader, JSON)
Incremental, exactly-once ingest of `landing/pos/`. Keeps everything as it arrived (duplicates included) plus source-file lineage.

In [0]:
%run ./_common

In [0]:
NB = "10_bronze_pos"
TGT = f"{CAT}.bronze.pos_events"
CP = f"{CKPT}/bronze_pos"

def run():
    before = table_count(TGT)
    q = (spark.readStream.format("cloudFiles")
           .option("cloudFiles.format", "json")
           .option("cloudFiles.schemaLocation", f"{CP}/schema")
           .option("cloudFiles.inferColumnTypes", "true")
           .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
           .option("rescuedDataColumn", "_rescued_data")
           .load(f"{LANDING}/pos/")
           .select("*",
                   F.col("_metadata.file_path").alias("_source_file"),
                   F.col("_metadata.file_modification_time").alias("_file_modified_at"),
                   F.current_timestamp().alias("_ingested_at"))
           .writeStream
           .option("checkpointLocation", CP)
           .option("mergeSchema", "true")
           .trigger(availableNow=True)
           .toTable(TGT))
    q.awaitTermination()
    print(f"{table_count(TGT) - before:,} rows ingested (total {table_count(TGT):,})")

with_schema_retry(run)

In [0]:
rescued = spark.table(TGT).where("_rescued_data IS NOT NULL").count()
dq_check("pos_rescued_rows", rescued, rescued == 0, "0")
dq_finish(NB)